The following exercises are meant to be solved by gathering the bash commands incrimentally in two scripts, one for ex 1.* the other for ex 2.* 

### Ex 1

1\.a Make a new directory called `students` in your home. Download a csv file with the list of students of this lab from [here](https://www.dropbox.com/s/867rtx3az6e9gm8/LCP_22-23_students.csv) (use the `wget` command) and copy that to `students`. First check whether the file is already there

1\.b Make two new files, one containing the students belonging to PoD, the other to Physics.

1\.c For each letter of the alphabet, count the number of students whose surname starts with that letter. 

1\.d Find out which is the letter with most counts.

1\.e Assume an obvious numbering of the students in the file (first line is 1, second line is 2, etc.), group students "modulo 18", i.e. 1,19,37,.. 2,20,38,.. etc. and put each group in a separate file  

In [5]:
%%bash
pwd
git branch --show-current
git status --short

/mnt/g/Endless Word of PHYSICS/Master's Dgree/Università di Padova/LABORATORY OF COMPUTATIONAL PHYSICS_A/LaboratoryOfComputationalPhysics_Y8
lab3-bash-exam
 M .ipynb_checkpoints/Untitled-checkpoint.ipynb
 M 00_introduction.ipynb
 M 00ex_introduction.ipynb
 M 01_Fundamentals.ipynb
 M 01ex_Fundamentals.ipynb
 M 02_NumberRepresentation.ipynb
 M 02ex_NumberRepresentation.ipynb
 M 03_Bash.ipynb
 M 03ex_Bash.ipynb
 M 04_Numpy.ipynb
 M 04ex_Numpy.ipynb
 M 05_OSEMN.ipynb
 M 05ex_OSEMN.ipynb
 M 06_Pandas.ipynb
 M 06ex_Pandas.ipynb
 M 07_Visualization.ipynb
 M 07ex_Visualization.ipynb
 M 08_LinearAlgebra.ipynb
 M 08ex_LinearAlgebra.ipynb
 M 09_Algorithms.ipynb
 M 09ex_Algorithms.ipynb
 M 10_MonteCarlo.ipynb
 M 10ex_MonteCarlo.ipynb
 M README.md
 M README_GitInstructions.md
 M Untitled.ipynb
 M cal_housing.data
 M credit_card.dat
 M populations.txt
 M user_data.json
?? .ipynb_checkpoints/03_Bash-checkpoint.ipynb
?? .ipynb_checkpoints/03ex_Bash-checkpoint.ipynb
?? .ipynb_checkpoints/EXLCPA-checkpo

In [1]:
%%writefile ex3_1_students.sh
#!/bin/bash

set -e

# Exercise 3.1 — Final clean exam-style solution

# 1.a Create students directory in home
mkdir -p "$HOME/students"

# Define file and URL
file="$HOME/students/LCP_22-23_students.csv"
url="https://www.dropbox.com/s/867rtx3az6e9gm8/LCP_22-23_students.csv?dl=1"

# Download file only if it is not already there
if [ -f "$file" ]; then
    echo "CSV file already exists."
else
    echo "CSV file not found. Downloading..."

    if command -v wget >/dev/null 2>&1; then
        wget -O "$file" "$url"
    else
        curl -L -o "$file" "$url"
    fi
fi

# 1.b Create files for PoD and Physics students
awk -F',' 'NR > 1 && $4 == "PoD" {print $0}' "$file" > "$HOME/students/students_PoD.csv"
awk -F',' 'NR > 1 && $4 == "Physics" {print $0}' "$file" > "$HOME/students/students_Physics.csv"

# 1.c Count students whose surname starts with each letter
> "$HOME/students/surname_counts.txt"

for letter in {A..Z}
do
    awk -F',' -v letter="$letter" '
    NR > 1 {
        first_letter = toupper(substr($1, 1, 1))
        if (first_letter == letter) {
            count++
        }
    }
    END {
        print letter, count + 0
    }
    ' "$file" >> "$HOME/students/surname_counts.txt"
done

# 1.d Find the most common surname starting letter
sort -k2,2nr "$HOME/students/surname_counts.txt" | head -n 1 > "$HOME/students/most_common_letter.txt"

# 1.e Split students into 18 groups using modulo 18
awk -F',' '
NR > 1 {
    group = ((NR - 2) % 18) + 1
    print $0 > ENVIRON["HOME"] "/students/group_" group ".txt"
}
' "$file"

echo "Exercise 3.1 completed."

Overwriting ex3_1_students.sh


In [2]:
!bash -lc 'bash ex3_1_students.sh'

CSV file already exists.
Exercise 3.1 completed.


In [3]:
!bash -lc 'ls "$HOME/students"/group_*.txt | wc -l'

18


In [4]:
!bash -lc 'cat "$HOME/students/most_common_letter.txt"'

B 13


In [5]:
!bash -lc 'wc -l "$HOME/students/students_PoD.csv"'

67 /c/Users/sisto/students/students_PoD.csv


In [6]:
!bash -lc 'wc -l "$HOME/students/students_Physics.csv"'

5 /c/Users/sisto/students/students_Physics.csv


### Ex 2

2.a Make a copy of the file `data.csv` removing the metadata and the commas between numbers; call it `data.txt`

2\.b How many even numbers are there?

2\.c Distinguish the entries on the basis of `sqrt(X^2 + Y^2 + Z^2)` is greater or smaller than `100*sqrt(3)/2`. Count the entries of each of the two groups 

2\.d Make `n` copies of data.txt (with `n` an input parameter of the script), where the i-th copy has all the numbers divided by i (with `1<=i<=n`).

In [7]:
!bash -lc 'ls -lh data.csv'

-rw-r--r-- 1 sisto 197609 120 Jun 11 13:26 data.csv


In [ ]:
%%writefile data.csv
# metadata: example file for practice
# columns are X,Y,Z
X,Y,Z
10,20,30
50,50,50
100,0,0
90,90,90
2,4,6
3,5,7

In [8]:
%%writefile ex3_2_data.sh
#!/bin/bash

set -e

# Exercise 3.2 — Final clean Bash solution

# n is the input parameter
n="$1"

# Check input parameter
if [ -z "$n" ]; then
    echo "Usage: bash ex3_2_data.sh n"
    exit 1
fi

# Input and output files
input="data.csv"
output="data.txt"

# Check that data.csv exists
if [ ! -f "$input" ]; then
    echo "Error: data.csv not found."
    exit 1
fi

# 2.a Remove metadata and commas; create data.txt
# Keep only rows where the first three fields are numbers
awk -F',' '
$1 ~ /^-?[0-9]+(\.[0-9]+)?$/ &&
$2 ~ /^-?[0-9]+(\.[0-9]+)?$/ &&
$3 ~ /^-?[0-9]+(\.[0-9]+)?$/ {
    print $1, $2, $3
}
' "$input" > "$output"

# 2.b Count even numbers in data.txt
awk '
{
    for (i = 1; i <= NF; i++) {
        if ($i == int($i) && $i % 2 == 0) {
            count++
        }
    }
}
END {
    print count
}
' "$output" > even_count.txt

# 2.c Split rows according to distance from origin
awk '
BEGIN {
    threshold = 100 * sqrt(3) / 2
}
{
    x = $1
    y = $2
    z = $3

    distance = sqrt(x*x + y*y + z*z)

    if (distance > threshold) {
        print $0 > "data_greater.txt"
        greater++
    } else {
        print $0 > "data_smaller.txt"
        smaller++
    }
}
END {
    print "greater", greater + 0 > "distance_counts.txt"
    print "smaller_or_equal", smaller + 0 >> "distance_counts.txt"
}
' "$output"

# 2.d Make n copies of data.txt
# In the i-th copy, all numbers are divided by i
for ((i = 1; i <= n; i++))
do
    awk -v divisor="$i" '
    {
        for (j = 1; j <= NF; j++) {
            printf "%g", $j / divisor

            if (j < NF) {
                printf " "
            } else {
                printf "\n"
            }
        }
    }
    ' "$output" > "data_divided_by_${i}.txt"
done

echo "Exercise 3.2 completed."


Overwriting ex3_2_data.sh


In [9]:
!bash -lc 'bash ex3_2_data.sh 5'

Exercise 3.2 completed.


In [10]:
!bash -lc 'cat data.txt'

10 20 30
50 50 50
100 0 0
90 90 90
2 4 6
3 5 7


In [11]:
!bash -lc 'cat distance_counts.txt'

greater 2
smaller_or_equal 4


In [12]:
!bash -lc 'ls data_divided_by_*.txt'

data_divided_by_1.txt
data_divided_by_2.txt
data_divided_by_3.txt
data_divided_by_4.txt
data_divided_by_5.txt
